<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Core Diagnostic 3 - Perturbative Method
---

This notebook groups the diagnostic plots for the perturbative core method used by TPeanuts. The automated checks live in `tpeanuts.core.perturbative.test.test0_perturbative_models`, `tpeanuts.core.perturbative.test.test1_perturbative_spectral`, and `tpeanuts.core.perturbative.test.test2_perturbative_evolutor`; here we keep the visual checks that help interpret the model interface, spectral decomposition, and perturbative segment operator.

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: traceless Hamiltonian, spectral projectors, and perturbative evolution |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Spectral-Decomposition) | **Spectral Decomposition**: trace split, eigenvalues, projectors, reconstruction |
| [4](#4.-Zeroth-Order-Evolutor) | **Zeroth-Order Evolutor**: spectral exponential and unitarity |
| [5](#5.-First-Order-Perturbative-Correction) | **First-Order Perturbative Correction**: residual-density scaling and masking |
| [?](#6.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Traceless Hamiltonian Split

For a three-flavour Hamiltonian $H$, the perturbative core separates the trace contribution from the traceless part,

$$
H = T + \frac{\operatorname{Tr}(H)}{3}\,I_3,
\qquad
\operatorname{Tr}(T)=0.
$$

The trace only adds a global phase to coherent evolution. Oscillation probabilities are insensitive to this global phase, but the full segment operator still carries it when the matrix exponential is constructed.

### 0.2 Spectral Projectors

The traceless Hermitian matrix $T$ admits the spectral decomposition

$$
T = \sum_{a=1}^{3} \lambda_a M_a,
$$

where $\lambda_a$ are real eigenvalues and $M_a$ are the corresponding projectors. The projectors satisfy

$$
\sum_a M_a = I_3,
\qquad
M_a M_b = \delta_{ab} M_a,
\qquad
M_a^\dagger = M_a.
$$

TPeanuts builds these objects from the traceless invariants and reuses them in the perturbative evolutor, avoiding a direct matrix exponential in the hot path.

### 0.3 Zeroth-Order Evolution

For a segment with average Hamiltonian $H$ and dimensionless length $L$, the constant-density contribution is

$$
U_0(L) = \sum_{a=1}^{3}
\exp\left[-i\left(\lambda_a + \frac{\operatorname{Tr}(H)}{3}\right)L\right] M_a.
$$

For a Hermitian Hamiltonian this operator should be unitary and numerically equivalent to $\exp(-iHL)$.

### 0.4 First-Order Perturbative Correction

If the density inside the segment is written as an average profile plus a residual perturbation, the first-order correction can be expressed in the spectral basis of the average Hamiltonian,

$$
U(L) \simeq U_0(L) + U_1(L).
$$

The correction $U_1$ depends on the integral of the residual density weighted by phase differences between spectral modes. In the implementation this information is supplied by the profile model through `residual_integral(la, lb)`. A constant-density segment has zero residual and therefore $U_1=0$.

### References

- C. Giunti and C. W. Kim, *Fundamentals of Neutrino Physics and Astrophysics*, Oxford University Press, 2007.
- S. M. Bilenky, *Introduction to the Physics of Massive and Mixed Neutrinos*, Springer, 2010.
- J. J. Sakurai and J. Napolitano, *Modern Quantum Mechanics*, Cambridge University Press, 2020: spectral decomposition and time-evolution operator.

## 1. Libraries

All imports are centralized here. The notebook uses the same plotting and output helpers as the rest of the diagnostic series.

In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import numpy as np
import torch

from tpeanuts.core.common.hamiltonian import hamiltonian_kinetic_reduced, hamiltonian_matter_reduced
from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.core.common.potential import matter_potential_cc
from tpeanuts.core.perturbative.evolutor import (
    evolutor_first_order,
    evolutor_perturbative_from_H,
    evolutor_perturbative_segment,
    evolutor_zero_order,
)
from tpeanuts.core.perturbative.spectral import (
    hamiltonian_spectral_data,
    hamiltonian_spectral_projectors_traceless,
    hamiltonian_traceless,
    hamiltonian_traceless_c0,
    hamiltonian_traceless_c1,
    hamiltonian_traceless_eigenvalues,
)
from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show, to_numpy
from tpeanuts.util.context import RuntimeContext

## 2. Paths and Configuration

### 2.1 Paths

The output directory follows the notebook relative location below `notebooks/`: diagnostic figures from this notebook are saved under `diagnostic/core`.

In [ ]:
config = load_notebook_config()
OUTPUT_DIR = config.output_dir("diagnostic", "core")
SHOW_PLOTS = config.show_plots

print(f"Repository root: {config.package_dir}")
print(f"Figure output directory: {OUTPUT_DIR}")

### 2.2 Configuration

The diagnostic uses a fixed Hermitian $3\times3$ Hamiltonian for pure spectral checks and a NuFIT-based Standard Model oscillation setup for the physical segment example. The reference matrix is intentionally small and non-diagonal so that the eigenvalues and projectors are non-trivial while remaining easy to inspect.

**Expected results**: the Hamiltonian should be Hermitian, the trace split should reconstruct the original matrix, and the physical profile should provide a constant-density case where the perturbative correction is zero.

In [ ]:
ctx = RuntimeContext.resolve(config.device, config.dtype)
CDTYPE = torch.complex128 if ctx.dtype == torch.float64 else torch.complex64

H_EXAMPLE = torch.tensor(
    [
        [1.0 + 0.0j, 0.2 + 0.1j, 0.05 - 0.03j],
        [0.2 - 0.1j, 2.0 + 0.0j, 0.3 + 0.2j],
        [0.05 + 0.03j, 0.3 - 0.2j, 3.0 + 0.0j],
    ],
    device=ctx.device,
    dtype=CDTYPE,
)
L_EXAMPLE = torch.tensor(0.37, device=ctx.device, dtype=ctx.dtype)

oscillation = PropagationConfig.oscillation_parameters_from_preset("_SM_NUFIT52_NO", context=ctx)
ENERGY_MEV = torch.tensor(1000.0, device=ctx.device, dtype=ctx.dtype)
ELECTRON_DENSITY_MOL_CM3 = torch.tensor(1.5, device=ctx.device, dtype=ctx.dtype)
SEGMENT_LENGTH = torch.tensor(0.37, device=ctx.device, dtype=ctx.dtype)

print("H Hermitian max error:", torch.max(torch.abs(H_EXAMPLE - H_EXAMPLE.conj().T)).item())
print("Example segment length:", float(L_EXAMPLE))
print("Physical energy [MeV]:", float(ENERGY_MEV))
print("Electron density [mol/cm^3]:", float(ELECTRON_DENSITY_MOL_CM3))

### 2.3 Local Helpers

The only local helper is a minimal profile model exposing the interface expected by the perturbative evolutor. It is a diagnostic model: it is used to turn the residual integral on and off and to scan its amplitude, not to represent a full physical density profile.

In [ ]:
class ToyResidualProfile:
    def __init__(self, *, average, potential, length, zero_mask, residual_scale=0.0, perturbation=False):
        self.average = torch.as_tensor(average, device=ctx.device, dtype=ctx.dtype)
        self.potential = torch.as_tensor(potential, device=ctx.device, dtype=ctx.dtype)
        self.length = torch.as_tensor(length, device=ctx.device, dtype=ctx.dtype)
        self.zero_mask = torch.as_tensor(zero_mask, device=ctx.device, dtype=torch.bool)
        self.residual_scale = torch.as_tensor(residual_scale, device=ctx.device, dtype=ctx.dtype)
        self.perturbation = torch.as_tensor(perturbation, device=ctx.device, dtype=torch.bool)

    def residual_integral(self, la, lb):
        del la, lb
        shape = torch.broadcast_shapes(self.length.shape, self.residual_scale.shape)
        scale = torch.broadcast_to(self.residual_scale, shape)
        return scale[..., None, None] * torch.ones((*shape, 3, 3), device=ctx.device, dtype=ctx.dtype)

    def has_perturbation(self):
        shape = torch.broadcast_shapes(self.length.shape, self.perturbation.shape)
        return torch.broadcast_to(self.perturbation, shape)


def matrix_abs_np(matrix):
    return np.abs(to_numpy(matrix))


def frobenius_norm(matrix):
    return torch.linalg.matrix_norm(matrix).detach().cpu().numpy()

## 3. Spectral Decomposition

### 3.1 Trace Split and Matrix Reconstruction

This plot shows the absolute values of the original Hamiltonian, its traceless component, and the reconstructed Hamiltonian obtained from $T + \operatorname{Tr}(H)I/3$.

**Expected results**: the reconstruction residual should be compatible with numerical precision, while the traceless matrix keeps the off-diagonal structure of the original Hamiltonian.

In [ ]:
T_EXAMPLE, TRACE_EXAMPLE = hamiltonian_traceless(H_EXAMPLE)
H_RECONSTRUCTED = T_EXAMPLE + TRACE_EXAMPLE * torch.eye(3, device=ctx.device, dtype=CDTYPE) / 3.0
RECONSTRUCTION_ERROR = torch.abs(H_RECONSTRUCTED - H_EXAMPLE)

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), constrained_layout=True)
for ax, matrix, title in zip(
    axes,
    [H_EXAMPLE, T_EXAMPLE, RECONSTRUCTION_ERROR],
    [r"$|H|$", r"$|T|$", r"$|H_{rec}-H|$"],
):
    im = ax.imshow(matrix_abs_np(matrix), cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_and_show("diagnostic3_fig3_1_trace_split.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max reconstruction error:", torch.max(RECONSTRUCTION_ERROR).item())

### 3.2 Traceless Invariants and Eigenvalues

The spectral implementation computes the eigenvalues of the traceless Hamiltonian and uses the invariants $c_0=-\operatorname{Tr}(T^3)/3$ and $c_1=-\operatorname{Tr}(T^2)/2$ in the projector construction.

**Expected results**: the three eigenvalues should sum to zero because $T$ is traceless.

In [ ]:
C0_EXAMPLE = hamiltonian_traceless_c0(T_EXAMPLE)
C1_EXAMPLE = hamiltonian_traceless_c1(T_EXAMPLE)
LAMBDA_EXAMPLE = hamiltonian_traceless_eigenvalues(T_EXAMPLE)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.axhline(0.0, color="black", lw=0.8)
ax.bar(["lambda 1", "lambda 2", "lambda 3"], to_numpy(LAMBDA_EXAMPLE.real), color=["C0", "C1", "C2"])
ax.set_ylabel("Eigenvalue")
ax.set_title("Eigenvalues of the traceless Hamiltonian")
fig.tight_layout()
save_and_show("diagnostic3_fig3_2_traceless_eigenvalues.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)

print("c0:", C0_EXAMPLE)
print("c1:", C1_EXAMPLE)
print("sum eigenvalues:", LAMBDA_EXAMPLE.sum())

### 3.3 Spectral Projectors

The projectors encode the eigenspaces of the traceless Hamiltonian. Their absolute matrix elements are shown side by side.

**Expected results**: the sum of the three projectors should be the identity matrix, and $\sum_a \lambda_a M_a$ should reconstruct $T$.

In [ ]:
PROJECTORS, LAMBDA_FROM_PROJECTORS, C1_FROM_PROJECTORS = hamiltonian_spectral_projectors_traceless(T_EXAMPLE)
PROJECTOR_SUM = PROJECTORS.sum(dim=0)
T_FROM_PROJECTORS = (LAMBDA_FROM_PROJECTORS[:, None, None] * PROJECTORS).sum(dim=0)

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), constrained_layout=True)
for i, ax in enumerate(axes):
    im = ax.imshow(matrix_abs_np(PROJECTORS[i]), cmap="magma", vmin=0.0, vmax=1.0)
    ax.set_title(rf"$|M_{i+1}|$")
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_and_show("diagnostic3_fig3_3_projectors.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max |sum(M)-I|:", torch.max(torch.abs(PROJECTOR_SUM - torch.eye(3, device=ctx.device, dtype=CDTYPE))).item())
print("max |sum(lambda M)-T|:", torch.max(torch.abs(T_FROM_PROJECTORS - T_EXAMPLE)).item())

## 4. Zeroth-Order Evolutor

### 4.1 Spectral Exponential versus Direct Matrix Exponential

The zero-order evolutor should reproduce the direct matrix exponential for a constant Hamiltonian.

**Expected results**: the absolute difference should remain at numerical precision, and the unitarity residual $U_0^\dagger U_0-I$ should be small.

In [ ]:
U0_EXAMPLE = evolutor_zero_order(H_EXAMPLE, L_EXAMPLE)
U0_DIRECT = torch.matrix_exp(-1j * H_EXAMPLE * L_EXAMPLE)
U0_ERROR = torch.abs(U0_EXAMPLE - U0_DIRECT)
UNITARITY_ERROR = U0_EXAMPLE.conj().T @ U0_EXAMPLE - torch.eye(3, device=ctx.device, dtype=CDTYPE)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
for ax, matrix, title in zip(
    axes,
    [U0_ERROR, UNITARITY_ERROR],
    [r"$|U_0-U_{exp}|$", r"$|U_0^\dagger U_0-I|$"],
):
    im = ax.imshow(matrix_abs_np(matrix), cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_and_show("diagnostic3_fig4_1_zero_order_errors.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max |U0-exp(-iHL)|:", torch.max(U0_ERROR).item())
print("max unitarity residual:", torch.max(torch.abs(UNITARITY_ERROR)).item())

### 4.2 Length Scan

The length scan checks the stability of the spectral exponential across a range of segment lengths.

**Expected results**: the matrix-exponential error and unitarity residual should stay near floating-point precision over the scan.

In [ ]:
length_grid = torch.linspace(0.0, 2.0, 81, device=ctx.device, dtype=ctx.dtype)
exp_errors = []
unitarity_errors = []
for length in length_grid:
    U0 = evolutor_zero_order(H_EXAMPLE, length)
    Uexp = torch.matrix_exp(-1j * H_EXAMPLE * length)
    exp_errors.append(float(torch.max(torch.abs(U0 - Uexp)).detach().cpu()))
    unitarity_errors.append(float(torch.max(torch.abs(U0.conj().T @ U0 - torch.eye(3, device=ctx.device, dtype=CDTYPE))).detach().cpu()))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.semilogy(to_numpy(length_grid), np.maximum(exp_errors, 1e-300), label=r"$\max|U_0-e^{-iHL}|$")
ax.semilogy(to_numpy(length_grid), np.maximum(unitarity_errors, 1e-300), label=r"$\max|U_0^\dagger U_0-I|$")
ax.set_xlabel("Dimensionless segment length")
ax.set_ylabel("Maximum absolute residual")
ax.set_title("Zeroth-order evolutor stability")
ax.legend()
fig.tight_layout()
save_and_show("diagnostic3_fig4_2_zero_order_length_scan.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)

## 5. First-Order Perturbative Correction

### 5.1 Residual Scaling

The toy profile model lets us scale the residual integral directly. This diagnostic isolates the response of the perturbative correction without mixing in geometry or density-model details.

**Expected results**: the norm of $U-U_0$ should grow linearly with the residual scale while the no-perturbation mask returns exactly $U_0$.

In [ ]:
residual_scales = torch.logspace(-8, -3, 60, device=ctx.device, dtype=ctx.dtype)
correction_norms = []
U0_BASE = evolutor_zero_order(H_EXAMPLE, L_EXAMPLE)
for scale in residual_scales:
    profile = ToyResidualProfile(
        average=1.0,
        potential=0.2,
        length=L_EXAMPLE,
        zero_mask=False,
        residual_scale=scale,
        perturbation=True,
    )
    U = evolutor_perturbative_from_H(H_EXAMPLE, L_EXAMPLE, profile)
    correction_norms.append(float(torch.linalg.matrix_norm(U - U0_BASE).detach().cpu()))

profile_masked = ToyResidualProfile(
    average=1.0,
    potential=0.2,
    length=L_EXAMPLE,
    zero_mask=False,
    residual_scale=1e-4,
    perturbation=False,
)
masked_delta = torch.linalg.matrix_norm(evolutor_perturbative_from_H(H_EXAMPLE, L_EXAMPLE, profile_masked) - U0_BASE)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.loglog(to_numpy(residual_scales), correction_norms, marker="o", ms=3.0, lw=1.2)
ax.set_xlabel("Toy residual integral scale")
ax.set_ylabel(r"$||U-U_0||_F$")
ax.set_title("First-order correction scaling")
fig.tight_layout()
save_and_show("diagnostic3_fig5_1_residual_scaling.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("masked perturbation norm:", float(masked_delta))

### 5.2 Constant Physical Segment

For a constant-density physical segment, the profile residual is zero. The segment-level evolutor should therefore match the manual zero-order operator built from the same kinetic and matter terms.

**Expected results**: the segment operator and manual $U_0$ should agree at numerical precision; a zero-length segment should return the identity matrix.

In [ ]:
potential = matter_potential_cc(ELECTRON_DENSITY_MOL_CM3, antinu=False, context=ctx)
constant_profile = ToyResidualProfile(
    average=ELECTRON_DENSITY_MOL_CM3,
    potential=potential,
    length=SEGMENT_LENGTH,
    zero_mask=False,
    residual_scale=0.0,
    perturbation=False,
)

U_SEGMENT = evolutor_perturbative_segment(oscillation, ENERGY_MEV, constant_profile)
H_KIN, KI = hamiltonian_kinetic_reduced(
    oscillation,
    ENERGY_MEV,
    oscillation.pmns.reduced(),
    return_ki=True,
)
H_PHYSICAL = H_KIN + hamiltonian_matter_reduced(oscillation, ELECTRON_DENSITY_MOL_CM3, context=ctx)
TRACE_PHYSICAL = (KI[..., 0] + KI[..., 1] + KI[..., 2] + potential).to(dtype=CDTYPE)
U_MANUAL = evolutor_zero_order(H_PHYSICAL, SEGMENT_LENGTH, trace_H=TRACE_PHYSICAL)

zero_profile = ToyResidualProfile(
    average=ELECTRON_DENSITY_MOL_CM3,
    potential=potential,
    length=torch.tensor(0.0, device=ctx.device, dtype=ctx.dtype),
    zero_mask=True,
    residual_scale=1.0e-4,
    perturbation=True,
)
U_ZERO = evolutor_perturbative_segment(oscillation, ENERGY_MEV, zero_profile)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
for ax, matrix, title in zip(
    axes,
    [U_SEGMENT - U_MANUAL, U_ZERO - torch.eye(3, device=ctx.device, dtype=CDTYPE)],
    [r"$|U_{segment}-U_0|$", r"$|U(L=0)-I|$"],
):
    im = ax.imshow(matrix_abs_np(matrix), cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_and_show("diagnostic3_fig5_2_segment_consistency.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
print("max segment residual:", torch.max(torch.abs(U_SEGMENT - U_MANUAL)).item())
print("max zero-length residual:", torch.max(torch.abs(U_ZERO - torch.eye(3, device=ctx.device, dtype=CDTYPE))).item())

## 6. Summary

The perturbative core separates the numerical work into two clear layers. First, `spectral.py` turns an average Hamiltonian into trace, traceless eigenvalues, and spectral projectors. The projector diagnostics verify the algebraic identities required by the method: completeness, orthogonality, and reconstruction of the traceless Hamiltonian.

Second, `evolutor.py` uses those spectral objects to build the constant-density operator and add the first-order density-residual correction only when the profile model reports a perturbation. The direct comparison with `torch.matrix_exp` checks the zero-order path, while the residual scan shows the expected linear scaling of the perturbative term.

Physically, the most important sanity check is that a constant-density segment has no residual correction: the perturbative segment reduces to the exact constant-density evolution operator. The zero-length convention is also explicit and returns the identity matrix, which is essential when geometry creates empty propagation segments.